## About this code sample
This notebook is adapted from an ongoing project of the Research Analytics Task Group I chair. The project collects data from multiple APIs and integrates them with locally maintained data on our institution's transformative publishing agreements. The resulting dataset supports a range of research analytics questions from library and campus stakeholders.

The workflow combines bibliographic, citation, open-access, publisher agreement, funding, and article processing charge (APC) data. I designed the data requirements, collection methods, integration logic, data processing and cleaning, and analyses based on the intended uses of the dataset. I developed the analytical workflow and code with assistance from generative AI tools for coding, debugging, optimization, and documentation. I reviewed, tested, and verified the code and resulting data throughout the project. A journal-level open-access classification component used in the project was developed by a colleague and is not included in this sample.

Section 6 demonstrates one use of the dataset: examining how OMB's changes to the Uniform Guidance related to publication and cost limitations may affect Vanderbilt research. The analysis identifies federally funded publications, examines how many are covered by publisher transformative agreements or are already open access, and estimates the APCs associated with federally funded research.

This sample retains selected validation and diagnostic steps because checking data coverage, identifiers, and merge quality is an important part of the analysis. Credentials, local source files, and some repetitive API-enrichment steps have been omitted.

The workflow was developed iteratively as new data sources and research questions emerged and continues to be refined. Its analyses depend on the coverage and quality of the underlying bibliographic, funding, open-access, publisher, and APC data. For example, missing funding information does not necessarily mean that a publication was unfunded; APC values are estimates rather than actual institutional payments (the estimation method is documented below); and Vanderbilt corresponding authors are currently identified using institution-name text because of data availability, which can miss records when affiliations are represented inconsistently. I am continuing to explore ways to improve the coverage and accuracy of the dataset.

## 1. Setup and configuration

In [ ]:
import json
import os
import re
import time
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
from urllib.parse import quote
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests
from dateutil import parser as dateutil_parser
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv()


In [ ]:
# Credentials are read from environment variables; no secrets are stored in the notebook.
API_KEY = os.getenv("API_KEY")
INST_TOKEN = os.getenv("INST_TOKEN")
UNPAYWALL_EMAIL = os.getenv("UNPAYWALL_EMAIL")
OPENALEX_EMAIL = os.getenv("OPENALEX_EMAIL")
OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY")  # optional for some OpenAlex calls

if not API_KEY or not INST_TOKEN:
    raise ValueError("Elsevier API credentials are missing.")
if not UNPAYWALL_EMAIL:
    raise ValueError("UNPAYWALL_EMAIL is missing.")
if not OPENALEX_EMAIL:
    raise ValueError("OPENALEX_EMAIL is missing.")

# Input/output files used by the integration stage.
TA_FILE = Path("TA_Pub_Mapping_Cumulative_Update.xlsx")
OUTPUT_CSV = Path("vanderbilt_publications_enriched.csv")
FINAL_OUTPUT = Path("vanderbilt_research_analytics_integrated.xlsx")

pipeline_start = time.time()


In [ ]:
# Track elapsed time
def print_stage_time(stage_name, start_time):
    elapsed = time.time() - start_time
    print(f"\n{stage_name} finished in:")
    print(f"  {elapsed:.1f} seconds")
    print(f"  {elapsed/60:.2f} minutes")
    print(f"  {elapsed/3600:.2f} hours\n")

# Request and checkpoint settings
TIMEOUT = 30
SLEEP_SHORT = 0.17  # Scopus rate limit: approximately 6-9 requests/second
SLEEP_LONG = 0.5    # Unpaywall

today_str = datetime.now(ZoneInfo("America/Chicago")).strftime("%Y-%m-%d")
CHECKPOINT_DIR = f"checkpoints_{today_str}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Shared session with retry/backoff
def make_session(total_retries=3, backoff_factor=0.5,
                 status_forcelist=(429, 500, 502, 503, 504)):
    s = requests.Session()
    retries = Retry(
        total=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(["GET", "HEAD", "OPTIONS"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries, pool_connections=100, pool_maxsize=100)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

session = make_session()
session.headers.update({
    "X-ELS-APIKey": API_KEY,
    "X-ELS-Insttoken": INST_TOKEN,
})

# Checkpoint helpers
def load_json_cache(path, default=None):
    if default is None:
        default = {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return default


def save_json_cache(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


enriched_cache_path = os.path.join(CHECKPOINT_DIR, "enriched_records.json")
unpaywall_cache_path = os.path.join(CHECKPOINT_DIR, "unpaywall_cache.json")

enriched_cache = load_json_cache(enriched_cache_path, {})
unpaywall_cache = load_json_cache(unpaywall_cache_path, {})


def checkpoint_all():
    """Persist retrieved article metadata and OA lookups."""
    save_json_cache(enriched_cache_path, enriched_cache)
    save_json_cache(unpaywall_cache_path, unpaywall_cache)


# Consistent GET wrapper
def safe_get(url, params=None, headers=None, timeout=TIMEOUT, session=session):
    try:
        resp = session.get(url, params=params, headers=headers, timeout=timeout)
    except requests.RequestException as e:
        return None, e
    return resp, None


## 2. Retrieve and enrich publication data
The production workflow uses Scopus Search and Abstract Retrieval, Scopus journal metrics, PlumX, Unpaywall, and OpenAlex. To keep this sample readable, the code below shows the core Scopus retrieval and article-level enrichment plus the OA lookup. The journal-metric and PlumX stages use the same API/caching pattern and are omitted here.


### 2.1 Scopus publication search


In [ ]:
# Retrieve Vanderbilt publications from the Scopus Search API
SEARCH_URL = "https://api.elsevier.com/content/search/scopus"

headers = {
    "Accept": "application/json"
}

# 29 affiliation IDs for Vanderbilt
affiliation_ids = [
    60003915, 60116489, 60011078, 60014604, 60010138, 60030769,
    60279549, 60015753, 60006923, 60032506, 60011566, 60014035,
    60075880, 60018566, 60014083, 60076111, 60027095, 60032071,
    60010235, 60138808, 60009842, 60029432, 60023490, 60024890,
    60010060, 60281113, 60006475, 60024644, 60021068
]

aff_query = " OR ".join([f"AF-ID({aid})" for aid in affiliation_ids])

query = f"({aff_query}) AND PUBYEAR > 2020"

# Use cursor pagination to retrieve the full result set
params = {
    "query": query,
    "count": 25,  # Scopus Search API maximum
    "cursor": "*"
}

records = []

stage_start = time.time()

while True:
    # Use the shared session with retries
    resp, err = safe_get(SEARCH_URL, params=params, headers=headers)
    if err:
        raise RuntimeError(f"Network error during Scopus search: {err}")
    # Raise non-network HTTP errors
    resp.raise_for_status()

    data = resp.json()

    entries = data["search-results"].get("entry", [])
    for e in entries:
        records.append({
            "eid": e.get("eid"),
            "doi": e.get("prism:doi"),
            "pub_year": e.get("prism:coverDate", "")[:4],
            "source_id": e.get("source-id"),
            "source_title": e.get("prism:publicationName"),
            "citedby_count": e.get("citedby-count")
        })

    # Advance to the next page
    next_cursor = data["search-results"].get("cursor", {}).get("@next")
    if not next_cursor:
        break

    params["cursor"] = next_cursor
    time.sleep(SLEEP_SHORT)

print_stage_time("Scopus Search", stage_start)

# Quick check
print(f"Retrieved {len(records):,} publication records.")
records[:3]


### 2.2 Article-level metadata


In [ ]:
# Retrieve article-level metadata from the Scopus Abstract Retrieval API
# Scopus EIDs provide broader coverage than DOIs in Scopus
ABSTRACT_URL = "https://api.elsevier.com/content/abstract/eid/"

headers = {
    "Accept": "application/xml"
}

# Namespaces
ns = {
    "dn": "http://www.elsevier.com/xml/svapi/abstract/dtd",
    "ce": "http://www.elsevier.com/xml/ani/common",
    "dc": "http://purl.org/dc/elements/1.1/",
    "prism": "http://prismstandard.org/namespaces/basic/2.0/",
    "xocs": "http://www.elsevier.com/xml/xocs/dtd"
}

# Normalize publication dates
def _normalize_date(s):
    if not s:
        return None
    try:
        return dateutil_parser.parse(s, fuzzy=True).strftime("%Y-%m-%d")
    except Exception:
        return None


stage_start = time.time()

enriched_records = []

for idx, rec in enumerate(records):
    eid = rec.get("eid")
    if not eid:
        continue

    # Reuse previously retrieved metadata when restarting the notebook
    if eid in enriched_cache:
        enriched_records.append(enriched_cache[eid])
        continue

    url = ABSTRACT_URL + eid + "?view=FULL"

    try:
        r, net_err = safe_get(url, headers=headers, timeout=TIMEOUT)
        if net_err:
            print(f"Network error for {eid}: {net_err}; skipping for now")
            time.sleep(SLEEP_SHORT)
            continue

        if r.status_code != 200:
            print(f"Failed for {eid}: {r.status_code}")
            continue

        root = ET.fromstring(r.content)


        # Core data
        core = root.find("dn:coredata", ns)
        if core is None:
            continue

        scopus_id = core.findtext("dc:identifier", namespaces=ns)
        if scopus_id:
            scopus_id = scopus_id.replace("SCOPUS_ID:", "")

        record = rec.copy()

        record.update({
            "scopus_id": scopus_id,
            "doi": core.findtext("prism:doi", namespaces=ns),
            "title": core.findtext("dc:title", namespaces=ns),
            "aggregation_type": core.findtext("prism:aggregationType", namespaces=ns),
            "subtype_description": core.findtext("dn:subtypeDescription", namespaces=ns),
            "publisher": core.findtext("dc:publisher", namespaces=ns),
            "openaccess": core.findtext("dn:openaccess", namespaces=ns),
            "openaccess_flag": core.findtext("dn:openaccessFlag", namespaces=ns),
            "citedby_count": core.findtext("dn:citedby-count", namespaces=ns),
            "source_id": core.findtext("dn:source-id", namespaces=ns)
        })

        issns = core.findall("prism:issn", namespaces=ns)
        record["issn"] = "; ".join(i.text for i in issns if i.text)

        # Publication date
        cover_date = (
            core.findtext("prism:coverDate", namespaces=ns)
            or root.findtext(".//prism:coverDate", namespaces=ns)
        )
        record["prism_cover_date"] = cover_date or None
        # Store a normalized ISO date for downstream analysis
        record["publication_date_iso"] = _normalize_date(cover_date) if cover_date else None


        # Authors
        authors = []
        for author in root.findall(".//dn:authors/dn:author", ns):
            name = author.findtext("ce:indexed-name", namespaces=ns)
            if name:
                authors.append(name)

        record["authors"] = "; ".join(authors)


        # Affiliations
        affiliations = set()
        for aff in root.findall("dn:affiliation", ns):
            name = aff.findtext("dn:affilname", namespaces=ns)
            if name:
                affiliations.add(name)

        record["affiliations"] = "; ".join(sorted(affiliations))


        # Correspondence
        corr = root.find(".//correspondence")

        if corr is not None:
            person = corr.find("person")
            record["corresponding_author"] = (
                person.findtext("ce:indexed-name", namespaces=ns)
                if person is not None else "not available"
            )

            aff = corr.find("affiliation")
            if aff is not None:
                orgs = aff.findall("organization")
                record["corresponding_institution"] = (
                    orgs[-1].text if orgs else "not available"
                )
                record["corresponding_city"] = aff.findtext("city")
                record["corresponding_source_text"] = aff.findtext(
                    "ce:source-text", namespaces=ns
                )
            else:
                record["corresponding_institution"] = "not available"
                record["corresponding_city"] = "not available"
                record["corresponding_source_text"] = "not available"
        else:
            record["corresponding_author"] = "not available"
            record["corresponding_institution"] = "not available"
            record["corresponding_city"] = "not available"
            record["corresponding_source_text"] = "not available"


        # Funding
        funding_texts = []
        for ft in root.findall(".//xocs:funding-text", ns):
            if ft.text:
                funding_texts.append(ft.text.strip())

        record["funding"] = "; ".join(funding_texts)

        agencies = set()
        for agency in root.findall(".//xocs:funding-agency", ns):
            if agency.text:
                agencies.add(agency.text.strip())

        record["funding_agencies"] = "; ".join(sorted(agencies))


        # Subject areas
        subjects = []
        for subj in root.findall(".//dn:subject-area", ns):
            label = subj.text
            code = subj.attrib.get("code")
            if label:
                subjects.append(f"{label} ({code})")

        record["subject_areas"] = "; ".join(subjects)

        enriched_records.append(record)
        enriched_cache[eid] = record
        if (idx + 1) % 100 == 0:
            checkpoint_all()

        time.sleep(SLEEP_SHORT)

    except Exception as e:
        print(f"Error processing {eid}: {e}")

print_stage_time("Abstract Retrieval", stage_start)

checkpoint_all()

enriched_records[:3]


### 2.3 Additional enrichment steps (omitted)
The production workflow also retrieves journal-level metrics from Scopus and research-impact metrics from PlumX. These stages follow the same API and caching pattern and are omitted here for readability.

### 2.4 Article open-access status

In [ ]:
# Retrieve article-level OA status from Unpaywall
BASE = "https://api.unpaywall.org/v2/"
DELAY = SLEEP_LONG

# Normalize DOIs
def normalize_doi(d):
    if not d:
        return None
    s = d.strip()
    s = s.replace("DOI:", "").replace("doi:", "")
    # remove common URL prefixes
    for prefix in ("https://doi.org/", "http://doi.org/", "https://dx.doi.org/", "http://dx.doi.org/"):
        if s.lower().startswith(prefix):
            s = s[len(prefix):]
            break
    return s

# Prepare unique normalized DOIs
normalized_all = [normalize_doi(r.get("doi")) for r in enriched_records]
unique_dois = sorted({d for d in normalized_all if d})
print("Unique normalized DOIs to query:", len(unique_dois))

stage_start = time.time()

oa_lookup = {}
for j, doi in enumerate(unique_dois):
    if doi in unpaywall_cache:
        oa_lookup[doi] = unpaywall_cache[doi]
        continue

    try:
        safe = quote(doi, safe="")
        r, net_err = safe_get(BASE + safe, params={"email": UNPAYWALL_EMAIL}, timeout=TIMEOUT)
        if net_err:
            unpaywall_cache[doi] = None
            oa_lookup[doi] = None
        else:
            if r.status_code == 200:
                jdata = r.json()
                unpaywall_cache[doi] = jdata.get("oa_status")
                oa_lookup[doi] = unpaywall_cache[doi]
            else:
                unpaywall_cache[doi] = None
                oa_lookup[doi] = None
    except Exception:
        unpaywall_cache[doi] = None
        oa_lookup[doi] = None

    if j % 500 == 0:
        checkpoint_all()

    time.sleep(DELAY)

print_stage_time("Unpaywall", stage_start)

checkpoint_all()

records_withoastatus = []
count_non_null = 0
for rec in enriched_records:
    norm = normalize_doi(rec.get("doi"))
    rec["oa_status"] = oa_lookup.get(norm) if norm else None
    if rec["oa_status"] is not None:
        count_non_null += 1
    records_withoastatus.append(rec)

records_withoastatus[:3]


### 2.5 Journal open-access classification (collaborative component)
The working project also includes an OpenAlex-based journal OA classifier that adds `journal_oa_status` (gold / hybrid / closed). That component was developed by a colleague and is not included in my individual code sample. Section 6.3 shows how its output is used when that field is available in the production dataset.

### 2.6 Save the enriched publication dataset

In [ ]:
# Arrange the fields created in the visible workflow
df = pd.DataFrame(records_withoastatus)

desired_order = [
    # Core identifiers
    "eid", "scopus_id", "doi", "title",

    # Publication information
    "pub_year", "aggregation_type", "subtype_description", "publisher",
    "openaccess", "openaccess_flag", "oa_status",
    "prism_cover_date", "publication_date_iso",

    # Authors and affiliations
    "authors", "affiliations", "corresponding_author",
    "corresponding_institution", "corresponding_city", "corresponding_source_text",

    # Subject and funding information
    "subject_areas", "funding", "funding_agencies",

    # Journal/source information and citations
    "source_id", "source_title", "issn", "citedby_count",
]

existing_columns = [col for col in desired_order if col in df.columns]
remaining_columns = [col for col in df.columns if col not in existing_columns]
df = df[existing_columns + remaining_columns]

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
checkpoint_all()

print(f"Saved file with {len(df):,} rows and {len(df.columns)} columns.")
print_stage_time("Visible retrieval workflow", pipeline_start)


## 3. Integrate publisher transformative-agreement data

Publisher agreement records are combined with the API-derived publication dataset. DOI normalization, duplicate review, and match diagnostics are retained because they affect the validity of the merged data.


In [ ]:
# Read and combine TA data
ta_file = TA_FILE

xls = pd.ExcelFile(ta_file)

output_sheets = [s for s in xls.sheet_names if s.endswith(" Output")]

# Read first sheet to establish columns
first_df = pd.read_excel(ta_file, sheet_name=output_sheets[0], header=2)
expected_columns = list(first_df.columns)

ta_dfs = []
sheet_summary = []

for sheet in output_sheets:

    df = pd.read_excel(ta_file, sheet_name=sheet, header=2)

    # Remove completely empty rows
    df = df.dropna(how="all")

    # Determine DOI column
    if "Article DOI" in df.columns:
        doi_col = "Article DOI"
    elif "DOI" in df.columns:
        doi_col = "DOI"
    else:
        doi_col = None

    # Record summary BEFORE any renaming
    sheet_summary.append({
        "Sheet": sheet,
        "Rows": len(df),
        "DOI column": doi_col,
        "Missing DOI": df[doi_col].isna().sum() if doi_col else pd.NA
    })

    # Rename DOI column if needed
    if doi_col == "DOI":
        df = df.rename(columns={"DOI": "Article DOI"})

    if list(df.columns) != expected_columns:
        raise ValueError(f"Column mismatch in sheet {sheet}")

    ta_dfs.append(df)

# Combine everything
ta_data = pd.concat(ta_dfs, ignore_index=True)

print(f"\nTA data contains {len(ta_data):,} rows from {len(output_sheets)} sheets.")

# Summary table
sheet_summary = pd.DataFrame(sheet_summary)

print("\nSheet summary:")
print(sheet_summary)


In [ ]:
# Read the API-derived publication dataset
full_data = pd.read_csv(
    OUTPUT_CSV,
    low_memory=False
)

# Remove unnamed columns
full_data = full_data.loc[
    :, ~full_data.columns.str.startswith("Unnamed")
]

print(f"Full data contains {len(full_data):,} rows.")
print(f"Full data contains {len(full_data.columns):,} columns.")


In [ ]:
# Audit Scopus EIDs before integration.
valid_eid = full_data["eid"].fillna("").str.fullmatch(r"2-s2\.0-\d+")
bad_eids = full_data.loc[~valid_eid, ["eid", "doi", "title"]].copy()

print(f"Malformed EIDs: {len(bad_eids):,}")
print(f"Valid EIDs: {valid_eid.sum():,}")
if not bad_eids.empty:
    display(bad_eids.head(20))

rows_removed = (~valid_eid).sum()
full_data = full_data.loc[valid_eid].reset_index(drop=True)
print(f"Removed {rows_removed:,} rows with malformed EIDs.")
print(f"Full data now contains {len(full_data):,} rows.")


In [ ]:
# Standardize identifiers and prepare fields for integration
def clean_doi(x):
    """Standardize DOI formatting."""
    if pd.isna(x):
        return pd.NA

    x = str(x)

    # Remove leading/trailing whitespace
    x = x.strip()

    # Remove all whitespace/newlines/tabs/non-breaking spaces
    x = re.sub(r"\s+", "", x)

    # Remove DOI URL if present
    x = re.sub(r"^https?://(dx\.)?doi\.org/", "", x, flags=re.I)

    # Remove leading "doi:"
    x = re.sub(r"^doi:", "", x, flags=re.I)

    # Lowercase
    x = x.lower()

    return x if x else pd.NA


# Standardize DOI columns
ta_data["Article DOI"] = ta_data["Article DOI"].apply(clean_doi)
full_data["doi"] = full_data["doi"].apply(clean_doi)


# Concatenate Funder Name 1–4
funder_cols = [
    "Funder Name 1",
    "Funder Name 2",
    "Funder Name 3",
    "Funder Name 4"
]

ta_data["TA data funder"] = (
    ta_data[funder_cols]
    .fillna("")
    .astype(str)
    .apply(
        lambda row: "; ".join(
            value.strip()
            for value in row
            if value.strip() and value.strip().lower() != "nan"
        ),
        axis=1
    )
    .replace("", pd.NA)
)


# Keep only the columns needed for the merge
ta_merge = ta_data[
    ["Article DOI", "TA data funder", "APC", "Corresponding Author Email"]
].copy()

ta_merge = ta_merge.rename(columns={
    "Article DOI": "TA data DOI",
    "APC": "TA data APC", "Corresponding Author Email": "TA corresponding author email"
})


# Check DOI coverage and overlap before merging

print("DOI match diagnostics:")

print(f"Rows in TA data: {len(ta_data):,}")
print(f"Rows in full data: {len(full_data):,}")

print(f"\nBlank DOIs in TA data: {ta_data['Article DOI'].isna().sum():,}")
print(f"Blank DOIs in full data: {full_data['doi'].isna().sum():,}")

# Number of rows with non-missing DOIs
ta_rows_with_doi = ta_data["Article DOI"].notna().sum()
full_rows_with_doi = full_data["doi"].notna().sum()

ta_dois = set(ta_data["Article DOI"].dropna())
full_dois = set(full_data["doi"].dropna())

print(
    f"\nUnique DOIs in TA data: "
    f"{len(ta_dois):,} "
    f"({len(ta_dois) / ta_rows_with_doi:.2%} of rows with DOIs)"
)

print(
    f"Unique DOIs in full data: "
    f"{len(full_dois):,} "
    f"({len(full_dois) / full_rows_with_doi:.2%} of rows with DOIs)"
)

matched_dois = ta_dois.intersection(full_dois)

print(f"\nUnique matching DOIs: {len(matched_dois):,}")

print(f"TA DOIs not found in full data: {len(ta_dois - full_dois):,}")
print(f"Full data DOIs not found in TA data: {len(full_dois - ta_dois):,}")


In [ ]:
# Find duplicated non-empty DOI values in the TA dataset
dup_dois = (
    ta_merge.loc[
        ta_merge["TA data DOI"].notna() &
        ta_merge["TA data DOI"].duplicated(keep=False),
        "TA data DOI"
    ]
    .unique()
)

# Show all rows for those duplicated DOIs
dup_rows = (
    ta_data[
        ta_data["Article DOI"].isin(dup_dois)
    ]
    .sort_values("Article DOI")
)

display(dup_rows)


In [ ]:
# Prepare one record per DOI for the merge
ta_merge_for_join = (
    ta_merge.loc[
        ta_merge["TA data DOI"].notna()
    ]
    .drop_duplicates(
        subset="TA data DOI",
        keep="first"
    )
    .copy()
)

# Keep the first occurrence of each non-missing DOI
# Retain all records with missing DOIs

full_data_dedup = pd.concat([
    full_data.loc[
        full_data["doi"].notna()
    ].drop_duplicates(
        subset="doi",
        keep="first"
    ),
    full_data.loc[
        full_data["doi"].isna()
    ]
], ignore_index=True)


# Check row counts before merging

print(f"Original full data rows: {len(full_data):,}")
print(f"Deduplicated full data rows: {len(full_data_dedup):,}")

print(f"Original TA rows: {len(ta_merge):,}")
print(f"TA rows used for merge: {len(ta_merge_for_join):,}")

print()

# Merge publisher data onto the publication dataset
combined_data = full_data_dedup.merge(
    ta_merge_for_join,
    how="left",
    left_on="doi",
    right_on="TA data DOI"
)


# Flag publications covered by a transformative agreement
combined_data["TA Covered"] = combined_data["TA data DOI"].notna().map({
    True: "Yes",
    False: "No"
})

# Move TA Covered immediately before TA data DOI

cols = combined_data.columns.tolist()

cols.insert(
    cols.index("TA data DOI"),
    cols.pop(cols.index("TA Covered"))
)

combined_data = combined_data[cols]

# Merge diagnostics
matched_full = combined_data["TA data DOI"].notna().sum()

matched_ta = (
    ta_merge_for_join["TA data DOI"]
    .isin(full_data_dedup["doi"])
    .sum()
)

ta_rows_with_doi = len(ta_merge_for_join)

ta_not_used = ta_rows_with_doi - matched_ta

print()

print(
    f"Matched {matched_full:,} of {len(full_data_dedup):,} "
    f"full data records "
    f"({matched_full / len(full_data_dedup):.2%})."
)

print(
    f"Matched {matched_ta:,} of {ta_rows_with_doi:,} "
    f"TA records with DOIs "
    f"({matched_ta / ta_rows_with_doi:.2%})."
)

print(
    f"{ta_not_used:,} records in the TA dataset "
    "were not included in this analysis."
)

print()

print(f"Combined dataset contains {len(combined_data):,} rows.")


## 4. Add APC estimates

OpenAlex is used to retrieve article processing charge (APC) information. When article-level paid APC data are unavailable, the workflow uses list-price data and then an imputed mean as a fallback. The source of each estimate is retained for transparency.


In [ ]:
# Retrieve APC data in DOI batches and checkpoint progress
# Estimate hierarchy: paid APC -> list APC -> imputed mean
# apc_source records which value was used.

# Settings
OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY")
BATCH_SIZE = 50          # 50 is very reliable
SAVE_FILE = "apc_enriched_checkpoint.csv"

# Load existing progress if available

if os.path.exists(SAVE_FILE):
    print("Loading saved progress...")
    df = pd.read_csv(SAVE_FILE)
else:
    print("Starting from original dataframe...")
    df = combined_data.copy()

# Standardize DOIs

df["doi_clean"] = df["doi"].apply(clean_doi)

# Create APC fields if needed

for col in ["apc_paid", "apc_list", "apc_used", "apc_source"]:
    if col not in df.columns:
        df[col] = pd.NA

# Identify records that still need APC data

remaining = df[
    df["apc_paid"].isna() &
    df["apc_list"].isna()
]

dois = remaining["doi_clean"].dropna().unique()

print(f"\nRemaining DOIs to query: {len(dois):,}")

# Retrieve APC data from OpenAlex in batches

for start in range(0, len(dois), BATCH_SIZE):

    batch = dois[start:start+BATCH_SIZE]

    filter_string = "|".join(batch)

    success = False

    for attempt in range(5):

        try:

            response = requests.get(
                "https://api.openalex.org/works",
                params={
                    "filter": f"doi:{filter_string}",
                    "per-page": BATCH_SIZE,
                    **({"api_key": OPENALEX_API_KEY} if OPENALEX_API_KEY else {})
                },
                timeout=60
            )

            response.raise_for_status()

            works = response.json()["results"]

            for work in works:

                doi = work.get("doi")

                if doi is None:
                    continue

                doi = doi.lower().replace("https://doi.org/", "")

                paid = None
                listed = None

                if work.get("apc_paid"):
                    paid = work["apc_paid"].get("value_usd")

                if work.get("apc_list"):
                    listed = work["apc_list"].get("value_usd")

                mask = df["doi_clean"] == doi

                df.loc[mask, "apc_paid"] = paid
                df.loc[mask, "apc_list"] = listed

            success = True
            break

        except requests.exceptions.RequestException as e:

            wait = 2 ** attempt

            print(
                f"Batch {start//BATCH_SIZE + 1}: "
                f"attempt {attempt+1}/5 failed."
            )
            print(e)
            print(f"Retrying in {wait} seconds...\n")

            time.sleep(wait)

    if not success:

        print(f"Skipping batch beginning at DOI {start}")

    # Save progress after each batch
    df.to_csv(SAVE_FILE, index=False)

    print(
        f"Processed "
        f"{min(start+BATCH_SIZE,len(dois)):,}"
        f"/{len(dois):,} remaining DOIs"
    )

# Select APC estimate: paid amount -> list price -> mean imputation

observed = df["apc_paid"].fillna(df["apc_list"])

mean_apc = observed.mean()
median_apc = observed.median()

print("\nSummary statistics")
print("------------------")
print(f"Mean APC (USD):   {mean_apc:,.2f}")
print(f"Median APC (USD): {median_apc:,.2f}")

df["apc_used"] = (
    df["apc_paid"]
      .fillna(df["apc_list"])
      .fillna(mean_apc)
)

df["apc_source"] = "imputed"

df.loc[df["apc_list"].notna(), "apc_source"] = "list"
df.loc[df["apc_paid"].notna(), "apc_source"] = "paid"

# Save results and report source distribution

df.drop(columns="doi_clean", inplace=True)

df.to_csv(SAVE_FILE, index=False)

print("\nAPC source distribution:")
print(df["apc_source"].value_counts())

print("\nAPC source percentages:")
print(
    df["apc_source"]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

print("\nAPC enrichment complete.")
print(f"Saved to: {SAVE_FILE}")


In [ ]:
# Use the APC-enriched dataset as the master analysis dataset
combined_data = df.copy()


## 5. Derive analysis variables


### 5.1 Vanderbilt corresponding-author flag

In [ ]:
combined_data["Vanderbilt Corresponding-Authored"] = (
    combined_data["corresponding_institution"]
    .fillna("")
    .str.contains("vanderbilt", case=False)
    .map({
        True: "Yes",
        False: "No"
    })
)

print(
    f"Identified "
    f"{(combined_data['Vanderbilt Corresponding-Authored'] == 'Yes').sum():,} "
    f"Vanderbilt corresponding-authored publications "
    f"({(combined_data['Vanderbilt Corresponding-Authored'] == 'Yes').mean()*100:.1f}%)."
)


### 5.2 Federal-funding classification

The production workflow uses a longer reference list of federal agencies, sub-agencies, programs, and common name variants. For readability, this sample shows a representative subset. The full reference list is maintained separately from the analytical logic.


In [ ]:
# Representative federal funder terms used in this code sample
federal_agencies = {
    "NIH": ["national institutes of health", "nih"],
    "NSF": ["national science foundation", "nsf"],
    "DOE": ["department of energy", "u.s. department of energy", "us department of energy", "doe"],
    "DOD": ["department of defense", "u.s. department of defense", "us department of defense", "dod"],
    "VA": ["department of veterans affairs", "veterans health administration", "va"],
    "NASA": ["national aeronautics and space administration", "nasa"],
    "EPA": ["environmental protection agency", "epa"],
    "USDA": ["united states department of agriculture", "u.s. department of agriculture", "usda"],
    "CDC": ["centers for disease control and prevention", "cdc"],
    "FDA": ["food and drug administration", "fda"],
}

ABBREVIATIONS = {"nih", "nsf", "doe", "dod", "va", "nasa", "epa", "usda", "cdc", "fda"}


In [ ]:
# Identify the matching federal agency

def matched_agency(row):

    text = " ".join([
        str(row.get("funding_agencies", "")),
        str(row.get("TA data funder", ""))
    ]).lower()

    for agency, variants in federal_agencies.items():

        for variant in variants:

            variant = variant.lower().strip()

            # Whole-word matching for agency abbreviations
            if variant in ABBREVIATIONS:

                if re.search(
                    rf"\b{re.escape(variant)}\b",
                    text
                ):
                    return agency, variant

            # Full organization names
            else:

                if variant in text:
                    return agency, variant

    return None, None


# Apply the classification

combined_data[["Matched Agency", "Matched Variant"]] = (
    combined_data.apply(
        lambda row: pd.Series(
            matched_agency(row)
        ),
        axis=1
    )
)

combined_data["Federally Funded"] = (
    combined_data["Matched Agency"]
        .notna()
        .map({
            True: "Yes",
            False: "No"
        })
)

print(
    f"Identified "
    f"{(combined_data['Federally Funded'] == 'Yes').sum():,} "
    f"federally funded publications "
    f"({(combined_data['Federally Funded'] == 'Yes').mean()*100:.1f}%)."
)


In [ ]:
# Overall funding availability and federal funding summary
# Missing funding information does not necessarily indicate that a publication was unfunded.

# Funding information available?
funding_available = (
    combined_data[["funding_agencies", "TA data funder"]]
    .fillna("")
    .astype(str)
    .apply(
        lambda row: row.str.strip().ne("").any(),
        axis=1
    )
)

# Build summary table
overall = pd.DataFrame({
    "Category": [
        "Funding Information Available",
        "Federally Funded",
        "Funding Available, Not Federal",
        "Funding Information Not Available"
    ],
    "Publications": [
        funding_available.sum(),
        (combined_data["Federally Funded"] == "Yes").sum(),
        (
            funding_available &
            (combined_data["Federally Funded"] == "No")
        ).sum(),
        (~funding_available).sum()
    ]
})

overall["Percent"] = (
    overall["Publications"]
    / len(combined_data)
    * 100
).round(1)

display(overall)


## 6. Example use case: federal research and open-access coverage

### 6.1 Federal funding and transformative-agreement coverage


In [ ]:
# Federal funding and TA coverage overview
summary_table = (
    combined_data
    .groupby("pub_year")
    .apply(lambda g: pd.Series({
        "Total Publications": len(g),

        "Federally Funded (n)":
            (g["Federally Funded"] == "Yes").sum(),

        "Federal Publications Not TA Covered (n)":
            (
                (g["Federally Funded"] == "Yes") &
                (g["TA Covered"] == "No")
            ).sum()
    }))
    .reset_index()
    .rename(columns={"pub_year": "Publication Year"})
)

summary_table["Federally Funded (%)"] = (
    summary_table["Federally Funded (n)"]
    / summary_table["Total Publications"]
    * 100
).round(1)

summary_table["Federal Publications Not TA Covered (%)"] = (
    summary_table["Federal Publications Not TA Covered (n)"]
    / summary_table["Federally Funded (n)"]
    * 100
).round(1)

summary_table = summary_table[
    [
        "Publication Year",
        "Total Publications",
        "Federally Funded (n)",
        "Federally Funded (%)",
        "Federal Publications Not TA Covered (n)",
        "Federal Publications Not TA Covered (%)"
    ]
]

summary_table


### 6.2 Article-level OA status for federally funded publications not covered by an agreement


In [ ]:
# Article-level OA status for federally funded publications not covered by a TA

fed_not_ta = combined_data[
    (combined_data["Federally Funded"] == "Yes") &
    (combined_data["TA Covered"] == "No")
].copy()

# Standardize OA status
fed_not_ta["oa_status"] = (
    fed_not_ta["oa_status"]
    .fillna("missing")
    .str.lower()
)

# Count publications by year and OA status
table = (
    fed_not_ta
    .groupby(["pub_year", "oa_status"])
    .size()
    .unstack(fill_value=0)
)

# Remove the "oa_status" label above the columns
table.columns.name = None

# Ensure all expected OA categories exist
oa_categories = ["gold", "hybrid", "closed", "green", "bronze", "missing"]

for col in oa_categories:
    if col not in table.columns:
        table[col] = 0

table = table[oa_categories]

# Total federally funded publications not covered by a TA
table["Federal Publications Not TA Covered (n)"] = table.sum(axis=1)

# Build summary table
summary = pd.DataFrame(index=table.index)

summary["Federal Publications Not TA Covered (n)"] = (
    table["Federal Publications Not TA Covered (n)"]
)

summary["Gold OA (n)"] = table["gold"]
summary["Gold OA (%)"] = (
    table["gold"]
    / table["Federal Publications Not TA Covered (n)"] * 100
).round(1)

summary["Hybrid OA (n)"] = table["hybrid"]
summary["Hybrid OA (%)"] = (
    table["hybrid"]
    / table["Federal Publications Not TA Covered (n)"] * 100
).round(1)

summary["Closed (n)"] = table["closed"]
summary["Closed (%)"] = (
    table["closed"]
    / table["Federal Publications Not TA Covered (n)"] * 100
).round(1)

summary["Green OA (n)"] = table["green"]
summary["Green OA (%)"] = (
    table["green"]
    / table["Federal Publications Not TA Covered (n)"] * 100
).round(1)

summary["Bronze OA (n)"] = table["bronze"]
summary["Bronze OA (%)"] = (
    table["bronze"]
    / table["Federal Publications Not TA Covered (n)"] * 100
).round(1)

summary["Missing OA Status (n)"] = table["missing"]
summary["Missing OA Status (%)"] = (
    table["missing"]
    / table["Federal Publications Not TA Covered (n)"] * 100
).round(1)

# Clean up row index
summary = (
    summary
    .reset_index()
    .rename(columns={"pub_year": "Publication Year"})
)

summary.columns.name = None

display(summary)


### 6.3 Journal-level OA status (using the collaborative classifier output)
This analysis uses `journal_oa_status`, the output of the colleague-developed component described above.

In [ ]:
# Journal-level OA status for federally funded publications not covered by a TA

if "journal_oa_status" not in combined_data.columns:
    print("journal_oa_status is created by the collaborative classifier and is not generated in this code sample.")
else:

    fed_not_ta = combined_data[
        (combined_data["Federally Funded"] == "Yes") &
        (combined_data["TA Covered"] == "No")
    ].copy()

    # Standardize journal OA status
    fed_not_ta["journal_oa_status"] = (
        fed_not_ta["journal_oa_status"]
        .fillna("missing")
        .str.lower()
        .str.strip()
    )

    # Count publications by year and journal OA status
    table = (
        fed_not_ta
        .groupby(["pub_year", "journal_oa_status"])
        .size()
        .unstack(fill_value=0)
    )

    # Remove the column name
    table.columns.name = None

    # Ensure all expected journal OA categories exist
    journal_categories = [
        "gold",
        "hybrid",
        "closed",
        "missing"
    ]

    for col in journal_categories:
        if col not in table.columns:
            table[col] = 0

    table = table[journal_categories]

    # Total federally funded publications not covered by a TA
    table["Federal Publications Not TA Covered (n)"] = table.sum(axis=1)

    # Build summary table
    summary = pd.DataFrame(index=table.index)

    summary["Federal Publications Not TA Covered (n)"] = (
        table["Federal Publications Not TA Covered (n)"]
    )

    summary["Gold Journal (n)"] = table["gold"]
    summary["Gold Journal (%)"] = (
        table["gold"]
        / table["Federal Publications Not TA Covered (n)"] * 100
    ).round(1)

    summary["Hybrid Journal (n)"] = table["hybrid"]
    summary["Hybrid Journal (%)"] = (
        table["hybrid"]
        / table["Federal Publications Not TA Covered (n)"] * 100
    ).round(1)

    summary["Closed Journal (n)"] = table["closed"]
    summary["Closed Journal (%)"] = (
        table["closed"]
        / table["Federal Publications Not TA Covered (n)"] * 100
    ).round(1)

    summary["Missing Journal OA Status (n)"] = table["missing"]
    summary["Missing Journal OA Status (%)"] = (
        table["missing"]
        / table["Federal Publications Not TA Covered (n)"] * 100
    ).round(1)

    # Clean up row index
    summary = (
        summary
        .reset_index()
        .rename(columns={"pub_year": "Publication Year"})
    )

    summary.columns.name = None

    display(summary)


### 6.4 APC estimates for Vanderbilt corresponding-authored, federally funded publications


In [ ]:
# Vanderbilt corresponding-authored federally funded publications

# Subset
vandy_federal = combined_data[
    (
        combined_data["corresponding_institution"]
        .fillna("")
        .str.contains("vanderbilt", case=False)
    )
    &
    (combined_data["Federally Funded"] == "Yes")
].copy()


# Yearly APC summary

summary = (
    vandy_federal
    .groupby("pub_year")
    .apply(lambda g: pd.Series({

        "Federal Publications": len(g),

        "TA Covered Publications":
            f"{(g['TA Covered'] == 'Yes').sum()} "
            f"({(g['TA Covered'] == 'Yes').mean()*100:.1f}%)",

        "Not TA Covered Publications":
            f"{(g['TA Covered'] == 'No').sum()} "
            f"({(g['TA Covered'] == 'No').mean()*100:.1f}%)",

        "Total Estimated APC":
            g["apc_used"].sum(),

        "Estimated APC Covered by TA":
            g.loc[
                g["TA Covered"] == "Yes",
                "apc_used"
            ].sum(),

        "Estimated APC Not Covered by TA":
            g.loc[
                g["TA Covered"] == "No",
                "apc_used"
            ].sum(),

        "Mean APC (TA Covered)":
            g.loc[
                g["TA Covered"] == "Yes",
                "apc_used"
            ].mean(),

        "Mean APC (Not TA Covered)":
            g.loc[
                g["TA Covered"] == "No",
                "apc_used"
            ].mean()

    }))
    .reset_index()
    .rename(columns={"pub_year": "Publication Year"})
)

# Add a total row

total_federal = len(vandy_federal)

ta_covered = (vandy_federal["TA Covered"] == "Yes").sum()
ta_not_covered = (vandy_federal["TA Covered"] == "No").sum()

total_row = pd.DataFrame({

    "Publication Year": ["Total"],

    "Federal Publications": [total_federal],

    "TA Covered Publications": [
        f"{ta_covered} ({ta_covered / total_federal * 100:.1f}%)"
    ],

    "Not TA Covered Publications": [
        f"{ta_not_covered} ({ta_not_covered / total_federal * 100:.1f}%)"
    ],

    "Total Estimated APC": [
        vandy_federal["apc_used"].sum()
    ],

    "Estimated APC Covered by TA": [
        vandy_federal.loc[
            vandy_federal["TA Covered"] == "Yes",
            "apc_used"
        ].sum()
    ],

    "Estimated APC Not Covered by TA": [
        vandy_federal.loc[
            vandy_federal["TA Covered"] == "No",
            "apc_used"
        ].sum()
    ],

    "Mean APC (TA Covered)": [
        vandy_federal.loc[
            vandy_federal["TA Covered"] == "Yes",
            "apc_used"
        ].mean()
    ],

    "Mean APC (Not TA Covered)": [
        vandy_federal.loc[
            vandy_federal["TA Covered"] == "No",
            "apc_used"
        ].mean()

    ]

})

summary = pd.concat(
    [summary, total_row],
    ignore_index=True
)

# Format APC columns
currency_cols = [
    "Total Estimated APC",
    "Estimated APC Covered by TA",
    "Estimated APC Not Covered by TA",
    "Mean APC (TA Covered)",
    "Mean APC (Not TA Covered)"
]

for col in currency_cols:
    summary[col] = summary[col].apply(
        lambda x: f"${x:,.0f}" if pd.notna(x) else ""
    )

display(summary)

# APC source summary

apc_source_summary = (
    vandy_federal
    .groupby("apc_source")
    .agg(
        Articles=("apc_source", "size")
    )
    .reset_index()
)

apc_source_summary["Percent"] = (
    apc_source_summary["Articles"]
    / apc_source_summary["Articles"].sum()
    * 100
).round(1)

display(apc_source_summary)


### 6.5 APC validation check
As a reasonableness check, estimated APCs for publications known to be covered by transformative agreements are compared with APC totals in the publisher agreement data.

In [ ]:
# Compare estimated APCs with known TA data
reported_ta_apc = combined_data["TA data APC"].sum()
print(f"Reported APC in TA data: ${reported_ta_apc:,.0f}")

estimated_ta_apc = combined_data.loc[
    combined_data["TA Covered"] == "Yes",
    "apc_used"
].sum()
print(f"Estimated APC for TA-covered publications: ${estimated_ta_apc:,.0f}")


## 7. Save analysis-ready data


In [ ]:
combined_data.to_excel(FINAL_OUTPUT, index=False)

fa_subset = combined_data.loc[
    (combined_data["Federally Funded"] == "Yes")
    & (combined_data["Vanderbilt Corresponding-Authored"] == "Yes")
].copy()

subset_output = FINAL_OUTPUT.with_name(FINAL_OUTPUT.stem + "_federal_subset.xlsx")
fa_subset.to_excel(subset_output, index=False)

print(f"Saved integrated dataset: {FINAL_OUTPUT}")
print(f"Saved federal subset: {subset_output}")
